# ArcGIS REST JSON → GeoPackage Converter

Menggabungkan hasil **batch download ArcGIS REST Query JSON** menjadi satu GeoPackage (.gpkg).

### Fitur
- Membaca `.json` dari **folder utama + semua subfolder**
- Otomatis mendeteksi **Point / MultiPoint / Polyline / Polygon**
- Membaca `spatialReference` dari metadata ArcGIS
- Menggabungkan seluruh batch `features`
- Menangani field yang berbeda antar batch
- Output: **GeoPackage (.gpkg)** - format utama yang aman untuk data besar


In [ ]:
# 1. Instalasi library
%pip install -q geopandas shapely pyproj fiona pyogrio tqdm pandas


In [1]:
# 2. Import
import os
import json
import glob
import warnings
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, MultiPoint, LineString, MultiLineString
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry import shape as shapely_shape
from pyproj import CRS
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("GeoPandas :", gpd.__version__)
print("Pandas    :", pd.__version__)


GeoPandas : 1.1.4
Pandas    : 2.3.3


In [2]:
# 3. KONFIGURASI
#
# Ganti INPUT_DIR dengan folder tempat hasil download JSON ArcGIS REST kamu.
# Contoh:
# INPUT_DIR = r"D:\RuangKita\arcgis_download"
# atau:
# INPUT_DIR = "/home/user/arcgis_download"

INPUT_DIR = r"D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI"

# Folder output
OUTPUT_DIR = r"D:\DATABASE\PUSAT\ATRBPN\LSD"

# Nama dasar output
OUTPUT_NAME = "LSD_12_PROVINSI"

# Tambahkan path relatif file sumber ke atribut _source
ADD_SOURCE_FIELD = True

# Jika True, coba hapus fitur duplikat berdasarkan OBJECTID/FID/OID.
# Jika data memang mempunyai OBJECTID unik antar seluruh batch, ini berguna.
DEDUPLICATE = False

# Jika True, file JSON yang tidak punya "features" akan dilewati.
SKIP_NON_FEATURE_JSON = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input :", os.path.abspath(INPUT_DIR))
print("Output:", os.path.abspath(OUTPUT_DIR))


Input : D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI
Output: D:\DATABASE\PUSAT\ATRBPN\LSD


## 4. Fungsi utilitas

Parser berikut menangani dua bentuk yang umum:

1. **ArcGIS REST JSON**
   `{"features": [{"attributes": ..., "geometry": ...}]}`
2. **GeoJSON**
   `{"type": "FeatureCollection", "features": [...]}`

Geometri ArcGIS akan dikonversi berdasarkan bentuk koordinatnya.


In [3]:
def find_json_files(input_dir):
    """Cari semua JSON secara rekursif."""
    return sorted(Path(input_dir).rglob("*.json"))


def get_spatial_reference(data):
    """Ambil CRS dari metadata ArcGIS/GeoJSON."""
    sr = data.get("spatialReference")

    if isinstance(sr, dict):
        # Prioritas latestWkid lalu wkid
        wkid = sr.get("latestWkid") or sr.get("wkid")
        if wkid:
            try:
                return CRS.from_user_input(f"EPSG:{wkid}")
            except Exception:
                pass

        wkt = sr.get("wkt") or sr.get("wkt2")
        if wkt:
            try:
                return CRS.from_wkt(wkt)
            except Exception:
                pass

    # Beberapa GeoJSON menggunakan crs
    crs_obj = data.get("crs")
    if isinstance(crs_obj, dict):
        props = crs_obj.get("properties", {})
        name = props.get("name")
        if name:
            try:
                return CRS.from_user_input(name)
            except Exception:
                pass

    # GeoJSON modern default = WGS84
    if data.get("type") == "FeatureCollection":
        return CRS.from_epsg(4326)

    return None


def arcgis_geometry_to_shapely(geom):
    """Konversi geometry ArcGIS REST ke Shapely."""
    if not geom:
        return None

    # Point
    if "x" in geom and "y" in geom:
        return Point(geom["x"], geom["y"])

    # Multipoint
    if "points" in geom:
        return MultiPoint(geom["points"])

    # Polyline: ArcGIS = paths -> [path1, path2, ...]
    if "paths" in geom:
        paths = geom["paths"]
        lines = [LineString(path) for path in paths if len(path) >= 2]
        if not lines:
            return None
        return lines[0] if len(lines) == 1 else MultiLineString(lines)

    # Polygon: ArcGIS = rings -> [ring1, ring2, ...]
    # Polygon ArcGIS dapat memiliki outer/inner rings.
    # Shapely polygonization yang paling aman untuk data umum dilakukan
    # melalui orientasi ring + polygonize.
    if "rings" in geom:
        rings = geom["rings"]
        polygons = []

        # Gunakan polygon dari setiap ring. Untuk dataset sederhana,
        # ring pertama dapat menjadi shell; ring berikutnya yang berada
        # di dalam shell dianggap hole.
        valid_rings = []
        for ring in rings:
            if len(ring) >= 4:
                try:
                    p = Polygon(ring)
                    if not p.is_empty and p.is_valid:
                        valid_rings.append(p)
                except Exception:
                    pass

        if not valid_rings:
            return None

        # Cari ring yang bukan berada di dalam ring lain sebagai shell.
        shells = []
        holes = []
        for i, p in enumerate(valid_rings):
            contained = False
            for j, q in enumerate(valid_rings):
                if i != j and q.contains(p):
                    contained = True
                    break
            if contained:
                holes.append(p)
            else:
                shells.append(p)

        for shell in shells:
            shell_holes = [h.exterior.coords[:] for h in holes if shell.contains(h)]
            try:
                poly = Polygon(shell.exterior.coords[:], shell_holes)
                if not poly.is_empty:
                    polygons.append(poly)
            except Exception:
                polygons.append(shell)

        if not polygons:
            polygons = valid_rings

        return polygons[0] if len(polygons) == 1 else MultiPolygon(polygons)

    return None


def feature_to_record(feature, source_file):
    """Konversi satu feature ArcGIS/GeoJSON menjadi record + geometry."""
    # GeoJSON Feature
    if feature.get("type") == "Feature":
        props = dict(feature.get("properties") or {})
        geom = feature.get("geometry")
        geometry = shapely_shape(geom) if geom else None
    else:
        # ArcGIS REST Feature
        props = dict(feature.get("attributes") or {})
        geometry = arcgis_geometry_to_shapely(feature.get("geometry"))

    if ADD_SOURCE_FIELD:
        props["_source"] = str(source_file)

    props["_geometry_ok"] = geometry is not None
    props["_geometry_ok"] = bool(props["_geometry_ok"])

    return props, geometry


def read_arcgis_json(path):
    """Baca satu file dan kembalikan records, geometry, CRS."""
    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)

    # ArcGIS FeatureSet
    if isinstance(data, dict) and "features" in data:
        features = data.get("features") or []
    else:
        if SKIP_NON_FEATURE_JSON:
            return [], [], None, "non-feature-json"
        raise ValueError("JSON tidak memiliki property 'features'.")

    crs = get_spatial_reference(data)

    records = []
    geometries = []

    for feature in features:
        props, geometry = feature_to_record(feature, path)
        records.append(props)
        geometries.append(geometry)

    return records, geometries, crs, "ok"


## 5. Scan semua file JSON

In [4]:
json_files = find_json_files(INPUT_DIR)

print(f"Jumlah JSON ditemukan: {len(json_files)}")

if not json_files:
    raise FileNotFoundError(
        f"Tidak ditemukan file .json di: {os.path.abspath(INPUT_DIR)}"
    )

for p in json_files[:10]:
    print(" -", p)

if len(json_files) > 10:
    print(f" ... dan {len(json_files)-10} file lainnya")


Jumlah JSON ditemukan: 761
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0000001-0001000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0001001-0002000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0002001-0003000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0003001-0004000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0004001-0005000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0005001-0006000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0006001-0007000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0007001-0008000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0008001-0009000.json
 - D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\LSD_12_PROVINSI_0009001-0010000.json
 ... dan 751 file lainnya


## 6. Parse dan gabungkan seluruh batch

Notebook akan:
- membaca setiap JSON;
- mendeteksi CRS;
- mendeteksi tipe geometri secara otomatis dari hasil geometri;
- menggabungkan field yang berbeda antar batch;
- membuat satu `GeoDataFrame`.


In [5]:
all_records = []
all_geometries = []
file_report = []
crs_candidates = []

for path in tqdm(json_files, desc="Membaca JSON", unit="file"):
    try:
        records, geometries, crs, status = read_arcgis_json(path)

        all_records.extend(records)
        all_geometries.extend(geometries)

        if crs is not None:
            crs_candidates.append(crs)

        file_report.append({
            "file": str(path),
            "features": len(records),
            "crs": str(crs) if crs else None,
            "status": status
        })

    except Exception as e:
        file_report.append({
            "file": str(path),
            "features": 0,
            "crs": None,
            "status": f"ERROR: {e}"
        })

report_df = pd.DataFrame(file_report)

print("\n=== LAPORAN ===")
print("File diproses :", len(report_df))
print("Feature       :", len(all_records))
print("Error file    :", report_df["status"].astype(str).str.startswith("ERROR").sum())

report_df.head()


Membaca JSON:   0%|          | 0/761 [00:00<?, ?file/s]


=== LAPORAN ===
File diproses : 761
Feature       : 760273
Error file    : 0


,file,features,crs,status
0,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,1000,None,ok
1,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,1000,None,ok
2,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,1000,None,ok
3,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,1000,None,ok
4,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,1000,None,ok


In [6]:
# 7. Tentukan CRS
#
# Jika seluruh batch memakai CRS yang sama, CRS pertama digunakan.
# Jika ada beberapa CRS berbeda, tampilkan peringatannya.

unique_crs = []
for c in crs_candidates:
    try:
        c_norm = c.to_wkt()
        if c_norm not in [x.to_wkt() for x in unique_crs]:
            unique_crs.append(c)
    except Exception:
        pass

print("CRS yang ditemukan:")
for c in unique_crs:
    print(" -", c)

if not unique_crs:
    print("\nPERINGATAN: CRS tidak ditemukan. Default EPSG:4326 digunakan.")
    target_crs = CRS.from_epsg(4326)
elif len(unique_crs) == 1:
    target_crs = unique_crs[0]
else:
    print("\nPERINGATAN: ditemukan lebih dari satu CRS.")
    print("Semua fitur akan diasumsikan memakai CRS pertama.")
    print("Untuk dataset dengan CRS berbeda, sebaiknya pisahkan berdasarkan CRS.")
    target_crs = unique_crs[0]

print("\nCRS output:", target_crs)


CRS yang ditemukan:

PERINGATAN: CRS tidak ditemukan. Default EPSG:4326 digunakan.

CRS output: EPSG:4326


In [11]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# 8. Buat GeoDataFrame
gdf = gpd.GeoDataFrame(
    pd.DataFrame(all_records), geometry=all_geometries, crs=target_crs
)

# Bersihkan kolom helper jika tidak dibutuhkan
if "_geometry_ok" in gdf.columns:
    geometry_ok = gdf["_geometry_ok"].copy()


# Konversi MultiPolygon ke Polygon & isi None dengan Polygon kosong (tanpa menghapus data)
def force_to_polygon(geom):
    if geom is None or geom.is_empty:
        return Polygon()  # Mengisi None/kosong dengan Polygon kosong
    if geom.geom_type == "MultiPolygon":
        return max(
            geom.geoms, key=lambda g: g.area
        )  # Ambil polygon dengan area terbesar
    return geom


gdf["geometry"] = gdf["geometry"].apply(force_to_polygon)

# Output Info
print("Total fitur:", len(gdf))
print("Total kolom:", len(gdf.columns))
print("CRS:", gdf.crs)

# Tipe geometri otomatis
print("\nTipe geometri:")
print(gdf.geometry.geom_type.value_counts(dropna=False))

gdf.head()

Total fitur: 760273
Total kolom: 14
CRS: EPSG:4326

Tipe geometri:
Polygon    760273
Name: count, dtype: int64


,OBJECTID,WADMPR,WADMKK,LSD,FGSFRF,FKWS,LUASHA,CTKSWH,METADATA,Shape_Length,Shape_Area,_source,_geometry_ok,geometry
0,1,Lampung,Pesisir Barat,Lahan Sawah yang Dilindungi di luar Penetapan ...,000000,Belum Terdefinisi/Di luar Penetapan Kawasan Hutan,0.269582,Tidak Ada Cetak Sawah,,209.680809,2695.818507,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,True,"POLYGON ((11641636.578 -650238.1396, 11641610...."
1,2,Lampung,Pesisir Barat,Lahan Sawah yang Dilindungi di luar Penetapan ...,000000,Belum Terdefinisi/Di luar Penetapan Kawasan Hutan,0.363467,Tidak Ada Cetak Sawah,,360.617987,3634.666803,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,True,"POLYGON ((11641346.1544 -650161.1382, 11641345..."
2,3,Lampung,Pesisir Barat,Lahan Sawah yang Dilindungi di luar Penetapan ...,000000,Belum Terdefinisi/Di luar Penetapan Kawasan Hutan,25.920369,Tidak Ada Cetak Sawah,,4481.824080,259203.691844,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,True,"POLYGON ((11641879.9952 -649885.5816, 11641851..."
3,4,Lampung,Pesisir Barat,Lahan Sawah yang Dilindungi di luar Penetapan ...,000000,Belum Terdefinisi/Di luar Penetapan Kawasan Hutan,16.746916,Tidak Ada Cetak Sawah,,4335.768689,167469.162644,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,True,"POLYGON ((11642842.5761 -649965.5113, 11642809..."
4,5,Lampung,Pesisir Barat,Lahan Sawah yang Dilindungi di luar Penetapan ...,000000,Belum Terdefinisi/Di luar Penetapan Kawasan Hutan,0.346020,Tidak Ada Cetak Sawah,,320.894154,3460.196572,D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI\L...,True,"POLYGON ((11642342.3555 -649908.0713, 11642331..."


In [12]:
# 11. Opsional: deduplikasi
#
# Hanya aktif jika DEDUPLICATE = True.
# Prioritas field: OBJECTID, objectid, FID, fid, OID, oid.

if DEDUPLICATE:
    candidates = ["OBJECTID", "objectid", "FID", "fid", "OID", "oid"]
    id_field = next((c for c in candidates if c in gdf.columns), None)

    if id_field:
        before = len(gdf)
        gdf = gdf.drop_duplicates(subset=[id_field], keep="first").copy()
        print(f"Deduplikasi berdasarkan {id_field}: {before} -> {len(gdf)}")
    else:
        print("Tidak menemukan field ID standar. Deduplikasi dilewati.")
else:
    print("DEDUPLICATE=False → tidak ada fitur yang dihapus.")


DEDUPLICATE=False → tidak ada fitur yang dihapus.


## 12. Simpan GeoPackage (.gpkg)

GeoPackage adalah format utama yang aman untuk mempertahankan field dan geometri kompleks.


In [13]:
gpkg_path = os.path.join(OUTPUT_DIR, OUTPUT_NAME + ".gpkg")

gdf.to_file(
    gpkg_path,
    layer=OUTPUT_NAME,
    driver="GPKG",
    engine="pyogrio"
)

print("GeoPackage selesai:")
print(gpkg_path)


GeoPackage selesai:
D:\DATABASE\PUSAT\ATRBPN\LSD\LSD_12_PROVINSI.gpkg


## 13. Ringkasan hasil


In [ ]:
print("=" * 60)
print("RINGKASAN KONVERSI")
print("=" * 60)

print(f"Input JSON       : {len(json_files):,}")
print(f"Total fitur      : {len(gdf):,}")
print(f"Total atribut    : {len(gdf.columns):,}")
print(f"CRS              : {gdf.crs}")
print()

print("Geometry:")
print(gdf.geometry.geom_type.value_counts())

print("\nOutput:")
for f in sorted(Path(OUTPUT_DIR).glob("*.gpkg")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f" - {f.name:<40} {size_mb:,.2f} MB")


In [ ]:
# 14. Preview data
display(gdf.head(10))

print("\nBounds:")
print(gdf.total_bounds)


In [ ]:
# 15. Peta preview sederhana
import matplotlib.pyplot as plt

plot_gdf = gdf

if len(gdf) > 10000:
    plot_gdf = gdf.sample(10000, random_state=42)
    print("Menampilkan sample 10.000 fitur dari", len(gdf))

ax = plot_gdf.plot(figsize=(10, 8))
ax.set_title(f"{OUTPUT_NAME} - Preview")
ax.set_axis_off()
plt.show()
